## Project analysing government tendered contracts between 2015 and 2026 tendered by the Ministry of Defence.
The purpose is to analyse a number of government tendered contracts available publically from https://www.contractsfinder.service.gov.uk/ relating to the MOD. I aim to identify trends in spending around key inflection points, notable comparing pre and post full scale invasion of Ukraine in Feb 2022.

Data accounts for contracts published as awarded between 01/01/2015 and 23/09/2026

This markdown will contain the scripts used to intake, clean, and analyse data, alongside produce figures and visualisation.


## Setup and Imports

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import os
from pathlib import Path

csv_filenames = []
for dirname, _, filenames in os.walk("./contract_data"):
    for filename in filenames:
        csv_filenames.append(os.path.join(dirname, filename))
    
print(csv_filenames)

['./contract_data/notices(6).csv', './contract_data/notices(7).csv', './contract_data/notices(9).csv', './contract_data/notices(12).csv', './contract_data/notices(15).csv', './contract_data/notices(4).csv', './contract_data/notices.csv', './contract_data/notices(2).csv', './contract_data/notices(11).csv', './contract_data/notices(10).csv', './contract_data/notices(1).csv', './contract_data/notices(13).csv', './contract_data/notices(14).csv', './contract_data/notices(8).csv', './contract_data/notices(5).csv', './contract_data/notices(3).csv']


## Data Cleaning

In [ ]:
chunks = []
row_counts = {}

for path in csv_filenames:
    chunk = pd.read_csv(path, header=0)          
    chunk["source_file"] = Path(path).name        
    row_counts[Path(path).name] = len(chunk)
    chunks.append(chunk)
# parses each file as csv with header, concats them into filestream, creates dataframe at the end
combined = pd.concat(chunks, ignore_index=True)

# --- sanity checks ---
print(combined.shape)
print(pd.Series(row_counts))                       # rows contributed per file
print("duplicate notice IDs:", combined.duplicated(subset="Notice Identifier").sum())
print("column sets consistent:", len({tuple(c.columns) for c in chunks}) == 1)
print((combined.iloc[:, 0] == combined.columns[0]).sum())  # should be 0

combined.drop_duplicates(subset="Notice Identifier", inplace=True)  # remove duplicates
print("duplicate notice IDs after deduplication:", combined.duplicated(subset="Notice Identifier").sum())

combined.to_csv("combined_contract_data.csv", index=False)
print(combined.shape)

(10223, 44)
notices(6).csv     421
notices(7).csv     758
notices(9).csv     654
notices(12).csv    555
notices(15).csv    416
notices(4).csv     793
notices.csv        610
notices(2).csv     521
notices(11).csv    815
notices(10).csv    709
notices(1).csv     681
notices(13).csv    543
notices(14).csv    892
notices(8).csv     474
notices(5).csv     802
notices(3).csv     579
dtype: int64
duplicate notice IDs: 120
column sets consistent: True
0
duplicate notice IDs after deduplication: 0
(10103, 44)


Some duplicate values because I had slightly overlapping data periods. Shape of each file and entry consistent, number of entries after deduplication matches the number of entries retrieved in a search over the same data range on the original website. 

